# 05 - VQC (Variational Quantum Classifier)

Addestra e valuta il modello VQC (feature map + ansatz `RealAmplitudes`, ottimizzatore COBYLA) sui tre dataset in simulazione ideale, e traccia la curva di convergenza della funzione di costo (Figura `convergenza_vqc`, sezione "Configurazione sperimentale").

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from config import DATASET_CONFIGS, RANDOM_STATE, VQC_MAXITER
from src.pipeline import train_and_evaluate_vqc
from src.evaluation.plots import plot_confusion_matrix, plot_vqc_convergence

PROCESSED_DIR = Path.cwd().parent / "data" / "processed"
TABLES_DIR = Path.cwd().parent / "results" / "tables"
FIGURES_DIR = Path.cwd().parent / "results" / "figures"
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
rows = []
cost_histories = {}

for name, cfg in DATASET_CONFIGS.items():
    data = np.load(PROCESSED_DIR / f"{name}.npz")
    result, cost_history = train_and_evaluate_vqc(
        cfg["n_components"], cfg["feature_map_reps"], cfg["ansatz_reps"],
        cfg["entanglement"], data["X_train"], data["y_train"],
        data["X_test"], data["y_test"], maxiter=VQC_MAXITER,
        random_state=RANDOM_STATE,
    )
    result["dataset"] = name
    rows.append(result)
    cost_histories[name] = cost_history

    plot_confusion_matrix(
        result["confusion_matrix"], f"VQC - {name} (simulazione ideale)",
        FIGURES_DIR / f"confusion_matrix_vqc_{name}_ideale.png", cmap="Purples",
    )

vqc_results = pd.DataFrame(rows)
vqc_results.to_csv(TABLES_DIR / "vqc_ideal_results.csv", index=False)
vqc_results[["dataset", "accuracy", "precision", "recall", "f1_score",
             "fit_time_s", "n_circuits", "circuit_depth"]]

In [ ]:
from IPython.display import Image

vqc_convergence_path = FIGURES_DIR / "vqc_convergenza.png"
plot_vqc_convergence(cost_histories, vqc_convergence_path)
Image(filename=str(vqc_convergence_path))